## 1 Data Preparation

This notebook prepares the data in the required format for Omnilingual finetuning.

In [1]:
#!/usr/bin/env python3
"""
Convert Romansh TSV to Omnilingual Parquet with smaller fragments.
Run from the omnilingual-romansh directory.
"""

import os
import io
import sys
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import soundfile as sf
from tqdm import tqdm
from helpers import get_idiom_name_by_folder
from constants import FOLDER_NAMES

# ----------------------------------------------------------------------
# Configuration – adjust these!
# ----------------------------------------------------------------------
DATA_ROOT = "romansh-data"                     # Path to your data (relative to this script)
OUTPUT_ROOT = "./rm-dataset/version=0"         # Output Parquet dataset location
LANGUAGE_CODE = "roh_Latn"
BATCH_SIZE = 100                               # Number of rows per output file (smaller)
ROW_GROUP_SIZE = 50                            # Rows per row group (smaller)

# ----------------------------------------------------------------------
# Helper: compress audio to OGG
# ----------------------------------------------------------------------
def compress_audio_to_ogg(audio_array, sample_rate):
    buffer = io.BytesIO()
    sf.write(buffer, audio_array, sample_rate, format='ogg')
    return buffer.getvalue()

# ----------------------------------------------------------------------
# Import official conversion utility (from the copied repo)
# ----------------------------------------------------------------------
sys.path.insert(0, os.getcwd())                # add current dir to Python path
from omnilingual_romansh.workflows.dataprep.audio_tools import binary_to_list_int8
print("✅ Using official binary_to_list_int8")

# ----------------------------------------------------------------------
# Write a batch to Parquet
# ----------------------------------------------------------------------
def write_batch(records, out_dir, part_idx):
    binary_array = pa.array([r['audio_bytes'] for r in records], type=pa.binary())
    audio_bytes_list = binary_to_list_int8(binary_array)

    table = pa.Table.from_pydict({
        'text': [r['text'] for r in records],
        'audio_bytes': audio_bytes_list,
        'audio_size': [r['audio_size'] for r in records],
        'corpus': pa.array([r['corpus'] for r in records], type=pa.dictionary(pa.int32(), pa.string())),
        'split': pa.array([r['split'] for r in records], type=pa.dictionary(pa.int32(), pa.string())),
        'language': pa.array([r['language'] for r in records], type=pa.dictionary(pa.int32(), pa.string())),
    })

    out_path = os.path.join(out_dir, f"part-{part_idx}.parquet")
    pq.write_table(table, out_path, row_group_size=ROW_GROUP_SIZE)
    print(f"  Wrote {len(records)} rows to {out_path}")

# ----------------------------------------------------------------------
# Process one split (train/validation) for one idiom
# ----------------------------------------------------------------------
def process_split(idiom_folder, split_name):
    tsv_path = os.path.join(DATA_ROOT, idiom_folder, f"{split_name}.tsv")
    if not os.path.exists(tsv_path):
        print(f"⚠️ {tsv_path} not found, skipping.")
        return

    clips_dir = os.path.join(DATA_ROOT, idiom_folder, "clips")
    df = pd.read_csv(tsv_path, sep='\t')

    # Use human‑readable corpus name for directory structure
    corpus_name = get_idiom_name_by_folder(idiom_folder)
    out_dir = os.path.join(OUTPUT_ROOT, f"corpus={corpus_name}", f"split={split_name}", f"language={LANGUAGE_CODE}")
    os.makedirs(out_dir, exist_ok=True)

    batch_records = []
    part_idx = 0

    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"{corpus_name}/{split_name}"):
        audio_path = os.path.join(clips_dir, row['path'])
        try:
            audio, sr = sf.read(audio_path)

            if sr != 16000:
                # For production, resample here (you may use librosa.resample or torchaudio)
                # For now, we assume the audio is already 16 kHz.
                pass

            ogg_bytes = compress_audio_to_ogg(audio, sr)
            audio_size = len(audio)

            batch_records.append({
                'text': row['sentence'],
                'audio_bytes': ogg_bytes,
                'audio_size': audio_size,
                'corpus': corpus_name,
                'split': split_name,
                'language': LANGUAGE_CODE,
            })
        except Exception as e:
            print(f"Error processing {audio_path}: {e}")

        if len(batch_records) >= BATCH_SIZE:
            write_batch(batch_records, out_dir, part_idx)
            part_idx += 1
            batch_records = []

    if batch_records:
        write_batch(batch_records, out_dir, part_idx)

# ----------------------------------------------------------------------
# Main loop
# ----------------------------------------------------------------------
def main():
    print("="*60)
    print("Converting Romansh data to Omnilingual Parquet (small fragments)")
    print("="*60)
    print(f"Data root: {DATA_ROOT}")
    print(f"Output root: {OUTPUT_ROOT}")
    print(f"Batch size: {BATCH_SIZE}, Row group size: {ROW_GROUP_SIZE}")
    print("="*60)

    # Optional: clear existing dataset (uncomment if you want a fresh start)
    # import shutil
    # if os.path.exists(OUTPUT_ROOT):
    #     shutil.rmtree(OUTPUT_ROOT)

    for idiom_folder in FOLDER_NAMES:
        corpus_name = get_idiom_name_by_folder(idiom_folder)
        print(f"\n📂 Processing idiom: {corpus_name} (folder: {idiom_folder})")
        for split in ['train', 'validation', 'test']:   # add 'test' if you have it
            print(f"  Split: {split}")
            process_split(idiom_folder, split)

    print("\n✅ Conversion complete!")

if __name__ == "__main__":
    main()

✅ Using official binary_to_list_int8
Converting Romansh data to Omnilingual Parquet (small fragments)
Data root: romansh-data
Output root: ./rm-dataset/version=0
Batch size: 100, Row group size: 50

📂 Processing idiom: Surmiran (folder: rmsurmiran-cc-2021-12-23)
  Split: train


Surmiran/train:   2%|▏         | 103/6376 [00:04<04:45, 21.99it/s]

  Wrote 100 rows to ./rm-dataset/version=0/corpus=Surmiran/split=train/language=roh_Latn/part-0.parquet


Surmiran/train:   3%|▎         | 207/6376 [00:08<04:20, 23.66it/s]

  Wrote 100 rows to ./rm-dataset/version=0/corpus=Surmiran/split=train/language=roh_Latn/part-1.parquet


...

  Wrote 100 rows to ./rm-dataset/version=0/corpus=Sursilvan/split=validation/language=roh_Latn/part-5.parquet


Sursilvan/validation: 100%|██████████| 689/689 [00:38<00:00, 17.71it/s]


  Wrote 89 rows to ./rm-dataset/version=0/corpus=Sursilvan/split=validation/language=roh_Latn/part-6.parquet
  Split: test


Sursilvan/test: 100%|██████████| 94/94 [00:05<00:00, 18.16it/s]


  Wrote 94 rows to ./rm-dataset/version=0/corpus=Sursilvan/split=test/language=roh_Latn/part-0.parquet

✅ Conversion complete!
